# 실습 3. Boosting과 Stacking

## 데이터
- 분류: `sklearn.datasets.load_breast_cancer()`
- 회귀: `sklearn.datasets.load_diabetes()`

## 실습 목표
- GradientBoostingClassifier의 주요 하이퍼파라미터 비교
- HistGradientBoostingClassifier 학습과 평가
- StackingClassifier로 여러 모델의 예측 결합
- StackingRegressor로 회귀 모델 결합


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import StackingClassifier, StackingRegressor
from sklearn.metrics import classification_report, root_mean_squared_error, r2_score

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('분류 데이터:', X.shape, y.shape)
print('target names:', cancer.target_names)


분류 데이터: (569, 30) (569,)
target names: ['malignant' 'benign']


## 문제 1. GradientBoostingClassifier 기본 모델 학습

GradientBoostingClassifier를 학습하고 분류 성능을 확인하세요.

### 요구사항
- `n_estimators=100`
- `learning_rate=0.1`
- `max_depth=3`
- `random_state=42`
- 학습셋/평가셋 accuracy와 `classification_report()` 출력

### 실행 결과
```text
학습셋 accuracy: 약 1.0000
평가셋 accuracy: 약 0.9474 이상
classification_report가 출력됨
```


In [2]:
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_clf.fit(X_train, y_train)
gb_y_pred= gb_clf.predict(X_test)

print("학습셋 accuracy: 약", gb_clf.score(X_train, y_train))
print("평가셋 accuracy: 약", gb_clf.score(X_test, y_test)," 이상")
print(
    classification_report(
        y_test,
        gb_y_pred,
    )
)

학습셋 accuracy: 약 1.0
평가셋 accuracy: 약 0.956140350877193  이상
              precision    recall  f1-score   support

           0       0.97      0.90      0.94        42
           1       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## 문제 2. learning_rate와 n_estimators 비교

learning_rate와 n_estimators 조합별 점수를 비교하세요.

### 요구사항
- `learning_rate`: `[0.03, 0.1, 0.3]`
- `n_estimators`: `[50, 100, 200]`
- 결과 컬럼: `learning_rate`, `n_estimators`, `train_accuracy`, `test_accuracy`
- `test_accuracy` 기준 내림차순 정렬

### 실행 결과
```text
9개 조합의 결과 표가 출력됨
상위 조합은 test_accuracy가 약 0.95 이상으로 출력됨
```


In [ ]:
param_grid = {
    'learning_rate': [0.03, 0.1, 0.3],
    'n_estimators': [50, 100, 200],
}

## 문제 3. HistGradientBoostingClassifier 학습

HistGradientBoostingClassifier를 학습하고 평가셋 accuracy를 확인하세요.

### 요구사항
- `max_iter=100`
- `learning_rate=0.1`
- `max_leaf_nodes=15`
- `random_state=42`
- 학습셋/평가셋 accuracy 출력

### 실행 결과
```text
학습셋 accuracy와 평가셋 accuracy가 출력됨
평가셋 accuracy는 약 0.94 이상으로 출력됨
```


## 문제 4. StackingClassifier 구성과 비교

Logistic Regression, KNN, Decision Tree를 기본 모델로 사용해 StackingClassifier를 구성하세요.

### 요구사항
- Logistic Regression과 KNN은 `Pipeline([('scaler', StandardScaler()), ...])`로 구성
- Decision Tree는 `max_depth=4`, `random_state=42`
- final estimator는 `LogisticRegression(max_iter=1000)` 사용
- 개별 모델과 Stacking 모델의 train/test accuracy를 표로 비교

### 실행 결과
```text
logistic_regression, knn, decision_tree, stacking 4개 모델의 점수 표가 출력됨
```


## 문제 5. StackingRegressor 회귀 모델 학습

Diabetes 데이터로 StackingRegressor를 만들고 회귀 지표를 비교하세요.

### 요구사항
- 기본 모델: Ridge, KNeighborsRegressor, DecisionTreeRegressor
- Ridge와 KNN은 스케일링 Pipeline 사용
- final estimator는 `Ridge()` 사용
- 평가셋 R2와 RMSE를 표로 출력

### 실행 결과
```text
ridge, knn_regressor, decision_tree_regressor, stacking_regressor 4개 모델의 회귀 지표가 출력됨
```


In [ ]:
diabetes = load_diabetes(as_frame=True)
reg_X = diabetes.data
reg_y = diabetes.target

reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X,
    reg_y,
    test_size=0.2,
    random_state=42
)